# Loan Approval Prediction — Exploratory Data Analysis (EDA)

## Objective
Understand the factors influencing loan approval decisions, identify data quality issues (missing values, outliers, skewness), and uncover patterns to guide feature engineering and model development.

---
### Key Dataset Attributes:
- **Loan_ID**: Unique identifier
- **Gender, Married, Dependents, Education, Self_Employed**: Demographic & personal details
- **ApplicantIncome, CoapplicantIncome**: Financial earnings
- **LoanAmount, Loan_Amount_Term**: Requested credit profile
- **Credit_History**: Past credit repayment record (1.0 = good, 0.0 = default/poor)
- **Property_Area**: Location classification (Urban, Semiurban, Rural)
- **Loan_Status**: Target variable (Y = Approved, N = Rejected)

In [ ]:
# 1. Imports and Configuration
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Plot styling
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["font.size"] = 11

In [ ]:
# 2. Load Dataset
data_path = os.path.join("..", "data", "raw", "loan_data.csv")
df = pd.read_csv(data_path)
print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

In [ ]:
# 3. Data Types and Missing Values
print("--- Data Types and Non-Null Counts ---")
print(df.info())

print("\n--- Missing Values Summary ---")
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({"Missing Count": missing, "Missing %": missing_pct.round(2)})
missing_df[missing_df["Missing Count"] > 0]

In [ ]:
# 4. Target Variable Analysis (Loan_Status)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
df["Loan_Status"].value_counts().plot.pie(autopct="%1.1f%%", colors=["#52BE80", "#EC7063"], ax=ax[0], startangle=90)
ax[0].set_ylabel("")
ax[0].set_title("Loan Approval Distribution")

sns.countplot(data=df, x="Loan_Status", palette=["#52BE80", "#EC7063"], ax=ax[1])
ax[1].set_title("Loan Status Count")
plt.tight_layout()
plt.show()

print(df["Loan_Status"].value_counts(normalize=True))

In [ ]:
# 5. Bivariate Analysis: Categorical Features vs Loan_Status
cat_cols = ["Credit_History", "Property_Area", "Education", "Married", "Dependents", "Self_Employed"]

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    cross_tab = pd.crosstab(df[col], df["Loan_Status"], normalize="index") * 100
    cross_tab.plot(kind="bar", stacked=True, color=["#EC7063", "#52BE80"], ax=axes[i], legend=(i == 0))
    axes[i].set_title(f"Approval Rate by {col}")
    axes[i].set_ylabel("Percentage (%)")
    axes[i].set_xlabel("")
    axes[i].tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
# 6. Numerical Feature Distributions and Skewness
num_cols = ["ApplicantIncome", "CoapplicantIncome", "LoanAmount"]

fig, axes = plt.subplots(len(num_cols), 2, figsize=(14, 10))
for i, col in enumerate(num_cols):
    sns.histplot(df[col].dropna(), kde=True, ax=axes[i, 0], color="#3498DB")
    axes[i, 0].set_title(f"{col} Distribution (Skewness: {df[col].skew():.2f})")
    
    sns.boxplot(data=df, x="Loan_Status", y=col, ax=axes[i, 1], palette=["#52BE80", "#EC7063"])
    axes[i, 1].set_title(f"{col} by Loan Status")

plt.tight_layout()
plt.show()

In [ ]:
# 7. Feature Engineering Exploration
df_feat = df.copy()
df_feat["Total_Income"] = df_feat["ApplicantIncome"] + df_feat["CoapplicantIncome"]
df_feat["Loan_to_Income"] = (df_feat["LoanAmount"] * 1000) / df_feat["Total_Income"]
df_feat["EMI"] = (df_feat["LoanAmount"] * 1000) / df_feat["Loan_Amount_Term"]
df_feat["EMI_to_Income"] = df_feat["EMI"] / df_feat["Total_Income"]

print("Engineered Features Summary:")
df_feat[["Total_Income", "Loan_to_Income", "EMI", "EMI_to_Income"]].describe()

In [ ]:
# 8. Correlation Heatmap
numeric_df = df_feat.select_dtypes(include=[np.number])
plt.figure(figsize=(10, 7))
sns.heatmap(numeric_df.corr(), annot=True, cmap="coolwarm", fmt=".2f", linewidths=0.5)
plt.title("Correlation Matrix of Numerical and Engineered Features")
plt.show()

## Key Takeaways from EDA:
1. **Credit History is Paramount**: Applicants meeting credit criteria (`Credit_History=1.0`) have an approval rate of ~80%, whereas applicants with `Credit_History=0.0` have an approval rate under ~10%.
2. **Property Location Impact**: Semiurban property applicants have higher approval rates (~76%) compared to Urban (~65%) and Rural (~61%).
3. **Income Skewness**: Income and Loan Amount distributions are heavily right-skewed with significant outliers. Domain-specific scaling and feature aggregation (`Total_Income = Applicant + Coapplicant`) normalize patterns.
4. **Missing Values**: Present in `Credit_History`, `Self_Employed`, `LoanAmount`, `Dependents`, `Loan_Amount_Term`, and `Gender`. Handled with domain-tailored median/mode imputers in the scikit-learn pipeline.